# KGGEN_extended — Test Retrieval Run

This notebook runs **one user query** through the hybrid passage-retrieval pipeline
implemented in `retrieval_core.py` (functions) and `run_retrieval.py` (execution).

The pipeline scores every graph-connected passage with three normalized signals:

$$\text{score}(p) = 0.3 \cdot \text{textSim}(q, p) + 0.3 \cdot \text{meanTripletSim}(q, p) + 0.4 \cdot \text{ppr}(p)$$

where the PPR branch takes the top-k entities by query–entity cosine similarity,
builds the **unified k-hop subgraph** around them, runs **Personalized PageRank**
seeded on those entities and propagates the PPR mass to passages via `Source` edges.

The **indexes** and the **embeddings** are loaded from the local `cache/` directory
when they already exist; on the very first run they are created once and written to
that cache, so every later query (and later process) just loads them.


## Test query

| | |
|---|---|
| **Query** | `What is the relationship between salt consumption above 2 g NaCl/day and arterial blood pressure?` |
| **Real answer** | `an almost linear positive relationship with arterial blood pressure` |


In [ ]:
import importlib
from pathlib import Path

# The sibling modules are edited outside the notebook, so reload them to make
# sure this session uses their current code (retrieval_core first: run_retrieval
# imports its classes at import time).
import retrieval_core
import run_retrieval
importlib.reload(retrieval_core)
importlib.reload(run_retrieval)

from retrieval_core import RetrievalConfig, Retriever
from run_retrieval import print_result, print_metadata

# ── The query under test and its real answer ──
query = "What is the relationship between salt consumption above 2 g NaCl/day and arterial blood pressure?"
gold_answer = "an almost linear positive relationship with arterial blood pressure"

# ── Configuration (GPU 0; the encoder is BAAI/bge-large-en-v1.5) ──
config = RetrievalConfig(graph_path=Path("../data/graph/_aggregated_all/aggregated_graph.graphml"), 
                         passages_path=Path("../data/graph/_aggregated_all/passages.json"), 
                         cache_dir=Path("../data/cache"), 
                         hf_cache_dir=Path("/mnt/data/huggingface_cache"), gpu_index=0, verbose=True)
print(config.describe())

# Indexes + embeddings are loaded from `cache/` if present, otherwise they are
# created here (once) and cached for all later queries.
retriever = Retriever(config).warm_up()
print("Retriever ready.")


## Run the query

Executes the retrieval for the user query and prints the retrieved passages with
their per-branch score breakdown.


In [ ]:
result = retriever.retrieve(query)
print_result(result, max_chars=400)


## Retrieved passages

Full text and provenance metadata of each retrieved passage, ordered by the final
hybrid score.


In [ ]:
for rank, record in enumerate(result.passage_records, 1):
    breakdown = result.score_breakdown[rank - 1]
    meta = {k: v for k, v in (record.get('metadata') or {}).items()
            if k in ('file', 'title', 'country', 'heading', 'page_number', 'audience')}

    print('=' * 100)
    print(f"[{rank}] {record['passage_id']}")
    print(f"    final={breakdown['final_score']:.4f} | "
          f"text_sim={breakdown['text_sim']:.4f} | "
          f"triplet_sim={breakdown['triplet_sim']:.4f} | "
          f"ppr_score={breakdown['ppr_score']:.4f}")
    if meta:
        print("    " + " | ".join(f"{k}={v}" for k, v in meta.items()))
    print('-' * 100)
    print(record.get('text', ''))
    print()


---

**Real answer:** `an almost linear positive relationship with arterial blood pressure` —
read the retrieved passages above to check whether the answer is supported by the
retrieved context. No metrics are computed.


## Metadata-only access

The metadata of the retrieved passages can also be obtained **without** the passage
texts, through dedicated accessors:

| Accessor | Use |
|---|---|
| `result.get_metadata(keys=..., include_scores=...)` | metadata of an existing `RetrievalResult` |
| `result.metadata_json(...)` | same, serialized as a JSON string |
| `retriever.retrieve_metadata(query, ...)` | run a query and return only the metadata |
| `retriever.get_metadata(passage_ids)` | look up metadata for arbitrary passage ids (no encoder / GPU needed) |
| `retriever.get_metadata_index()` | the full `{passage_id: metadata}` mapping |
| `run_retrieval_metadata(query, ...)` | same, from the execution module (CLI: `--metadata-only`) |


In [ ]:
# ── Metadata of the retrieval above (passage texts are not used) ──
entries = result.get_metadata(keys=('file', 'heading', 'page_number'), include_scores=True)
print_metadata(entries)

print()
print("Full metadata of the top passage:")
for key, value in result.get_metadata()[0].items():
    print(f"  {key:12s} = {value}")


In [ ]:
from run_retrieval import run_retrieval_metadata

# Same query, metadata only, straight through the execution module
metadata = run_retrieval_metadata(query, config=config, verbose=False)
print(f"{len(metadata)} entries | fields: {list(metadata[0])}")

# Look up metadata for arbitrary passage ids
print(retriever.get_metadata(result.passage_ids[:2]))

# The full {passage_id: metadata} mapping is available too
metadata_index = retriever.get_metadata_index()
print(f"metadata index: {len(metadata_index)} passages, "
      f"e.g. {list(metadata_index.values())[0]}")


## Metadata stored on the graph's passage nodes

The aggregated graph stores each passage as a **node** with three attributes:
`type` (`'passage'`), `text` and `metadata` (a JSON string in the GraphML file,
parsed into a dict when the indexes are built). Those attributes are cached with
the indexes, so they can be read without touching `passages.json`:

| Accessor | Use |
|---|---|
| `result.get_graph_metadata(keys=..., include_scores=..., include_graph_attrs=...)` | graph metadata of an existing `RetrievalResult` |
| `result.graph_metadata_json(...)` | same, serialized as a JSON string |
| `result.passage_graph_attrs` | raw node attribute dicts (`type`, `text`, `metadata`) of the retrieved passages |
| `retriever.retrieve_graph_metadata(query, ...)` | run a query and return only the graph metadata |
| `retriever.get_graph_metadata(passage_ids)` | graph metadata for arbitrary passage ids (no encoder / GPU needed) |
| `retriever.graph_metadata_index()` | the full `{passage_id: metadata}` mapping of the graph's passage nodes |
| `run_retrieval_graph_metadata(query, ...)` | same, from the execution module (CLI: `--metadata-source graph`) |


In [ ]:
# ── Graph-node metadata of the retrieved passages ──
graph_entries = result.get_graph_metadata(keys=('file', 'heading', 'page_number'),
                                          include_scores=True)
print_metadata(graph_entries)

print()
print("Raw graph node attributes of the top passage:")
attrs = result.passage_graph_attrs[0]
print("  keys       :", list(attrs))
print("  type       :", attrs['type'])
print("  metadata   :", attrs['metadata'])
print("  text[:120] :", attrs['text'][:120], '...')


In [ ]:
from run_retrieval import run_retrieval_graph_metadata

# Same query, graph metadata only, straight through the execution module
# (`config` must be passed: without it the module would fall back to the
# default graph/passages paths).
graph_metadata = run_retrieval_graph_metadata(query, config=config, verbose=False)
print(f"{len(graph_metadata)} entries | fields: {list(graph_metadata[0])}")

# Graph metadata for arbitrary passage ids
print(retriever.get_graph_metadata(result.passage_ids[:2], keys=('file', 'page_number')))

# The full {passage_id: metadata} mapping of the graph's passage nodes
graph_index = retriever.graph_metadata_index()
print(f"graph metadata index: {len(graph_index)} passage nodes, "
      f"e.g. {list(graph_index.values())[0]}")
